In [1]:
import pandas as pd
import sqlite3
import numpy as np
import seaborn as sns
import matplotlib as plt

In [2]:
import pandas as pd

file_path = r"D:\PROJECTS\customer_data.csv"

df = pd.read_csv(file_path)

In [3]:
import sqlite3

conn = sqlite3.connect("customer_database.db")

In [4]:
df.to_sql("customer", conn, if_exists="replace", index=False)

3900

#### Q1. What is the total revenue generated by male vs. female customers?

In [5]:
def run_query(q):
    return pd.read_sql(q, conn)

In [6]:
run_query("""
SELECT gender, SUM(purchase_amount) AS revenue
FROM customer
GROUP BY gender
""")

,gender,revenue
0,Female,75191
1,Male,157890


#### Q2. Which customers used a discount but still spent more than the average purchase amount? 

In [7]:
run_query("""
SELECT customer_id, purchase_amount 
FROM customer 
WHERE discount_applied = 'Yes' AND purchase_amount >= (SELECT AVG(purchase_amount) FROM customer)
""")


,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


####  Q3. Which are the top 5 products with the highest average review rating?

In [8]:
run_query("""
SELECT 
    item_purchased,                     
    ROUND(AVG(review_rating),2) AS avg_rating   
FROM 
    customer                    
GROUP BY 
    item_purchased                     
ORDER BY 
    avg_rating DESC
LIMIT 
    5;
""")

,item_purchased,avg_rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,T-shirt,3.78


#### Q4. Compare the average Purchase Amounts between Standard and Express Shipping. 

In [9]:
run_query("""
SELECT
    shipping_type,                 
    ROUND(AVG(purchase_amount),2) AS avg_amt  
FROM
    customer  
    WHERE shipping_type IN ('Standard','Express')
GROUP BY
    shipping_type;    
    """)

,shipping_type,avg_amt
0,Express,60.48
1,Standard,58.46


#### Q5. Do subscribed customers spend more? Compare average spend and total revenue between subscribers and non-subscribers.

In [10]:
run_query("""SELECT subscription_status,
       COUNT(customer_id) AS total_customers,
       ROUND(AVG(purchase_amount),2) AS avg_spend,
       ROUND(SUM(purchase_amount),2) AS total_revenue
FROM customer
GROUP BY subscription_status
ORDER BY total_revenue,avg_spend DESC;
    """)

,subscription_status,total_customers,avg_spend,total_revenue
0,Yes,1053,59.49,62645.0
1,No,2847,59.87,170436.0


#### Q6. Which 5 products have the highest percentage of purchases with discounts applied?

In [11]:
run_query("""
SELECT item_purchased,
       ROUND(100.0 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END)/COUNT(*),2) AS discount_rate
FROM customer
GROUP BY item_purchased
ORDER BY discount_rate DESC
LIMIT 5;
""")

,item_purchased,discount_rate
0,Hat,50.00
1,Sneakers,49.66
2,Coat,49.07
3,Sweater,48.17
4,Pants,47.37


#### Q7. Segment customers into New, Returning, and Loyal based on their total number of previous purchases, and show the count of each segment.

In [12]:
run_query("""
with customer_type as (
SELECT customer_id, previous_purchases,
CASE 
    WHEN previous_purchases = 1 THEN 'New'
    WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
    ELSE 'Loyal'
    END AS customer_segment
FROM customer)

select customer_segment,count(*) AS "Number of Customers" 
from customer_type 
group by customer_segment;""")

,customer_segment,Number of Customers
0,Loyal,3116
1,New,83
2,Returning,701


#### Q8. What are the top 3 most purchased products within each category? 

In [13]:
run_query("""
WITH item_counts AS (
    SELECT category,
           item_purchased,
           COUNT(customer_id) AS total_orders,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM customer
    GROUP BY category, item_purchased
)
SELECT item_rank,category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <=3;""")

,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


#### Q9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?

In [14]:
run_query("""
SELECT subscription_status,
       COUNT(customer_id) AS repeat_buyers
FROM customer
WHERE previous_purchases > 5
GROUP BY subscription_status;
""")

,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


#### Q10. What is the revenue contribution of each age group? 

In [15]:
run_query(""" SELECT 
    age_group,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY age_group
ORDER BY total_revenue desc;
""")

,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763


In [16]:
df.to_csv("customer_final_data.xlsx", index=False)